In [3]:
# ============================================================
# 05_06_07_model_training.py — Phase D: Train, Compare, Tune
# Internship Project: NLP Sentiment Analysis (IMDB 50K)
# ============================================================
#
#  05 — Baseline  : Logistic Regression   (target: 88-90%)
#  06 — Comparison: NB, LogReg, LinearSVC (target: LinearSVC 92-93%)
#  07 — Final     : GridSearchCV + Save   (target: 93.8% F1)
#
# Run AFTER 04_feature_engineering.py
# ============================================================

import numpy as np
import pandas as pd
import joblib, os, time, warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LogisticRegression
from sklearn.naive_bayes     import MultinomialNB, BernoulliNB
from sklearn.svm             import LinearSVC
from sklearn.calibration     import CalibratedClassifierCV
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics         import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from sklearn.pipeline        import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Paths ─────────────────────────────────────────────────────
MODEL_DIR = "outputs/models"
PLOT_DIR  = "outputs/plots"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOT_DIR,  exist_ok=True)

RANDOM_STATE = 42

# ── Load Feature Matrices ────────────────────────────────────
def load_features():
    print("[LOAD] Loading feature matrices from disk …")
    X_train = joblib.load(f"{MODEL_DIR}/X_train_tfidf.pkl")
    X_test  = joblib.load(f"{MODEL_DIR}/X_test_tfidf.pkl")
    y_train = joblib.load(f"{MODEL_DIR}/y_train.pkl")
    y_test  = joblib.load(f"{MODEL_DIR}/y_test.pkl")
    print(f"       X_train: {X_train.shape}  X_test: {X_test.shape}")
    return X_train, X_test, y_train, y_test

def report_metrics(name, y_true, y_pred, y_proba=None):
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average='weighted')
    auc  = roc_auc_score(y_true, y_proba) if y_proba is not None else None
    print(f"\n  {'='*48}")
    print(f"  MODEL: {name}")
    print(f"  {'='*48}")
    print(f"  Accuracy  : {acc*100:.2f}%")
    print(f"  F1 (wtd)  : {f1*100:.2f}%")
    if auc: print(f"  ROC-AUC   : {auc:.4f}")
    print(classification_report(y_true, y_pred, target_names=['Negative','Positive']))
    return {'Model': name, 'Accuracy': acc, 'F1': f1, 'AUC': auc}

# ════════════════════════════════════════════════════════════
# SECTION 05 — BASELINE: Logistic Regression
# ════════════════════════════════════════════════════════════
def train_baseline(X_train, X_test, y_train, y_test):
    print("\n" + "="*55)
    print("  SECTION 05 — Baseline: Logistic Regression")
    print("="*55)

    t0  = time.time()
    lr  = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, C=1.0, solver='lbfgs')
    lr.fit(X_train, y_train)
    elapsed = time.time() - t0

    y_pred  = lr.predict(X_test)
    y_proba = lr.predict_proba(X_test)[:, 1]

    metrics = report_metrics("Logistic Regression (Baseline)", y_test, y_pred, y_proba)
    print(f"\n  Training time: {elapsed:.1f}s")

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred, display_labels=['Negative','Positive'],
        cmap='Blues', ax=ax
    )
    ax.set_title('Baseline: Logistic Regression — Confusion Matrix', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/06_baseline_confusion_matrix.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ Saved: 06_baseline_confusion_matrix.png")

    # Save baseline
    joblib.dump(lr, f"{MODEL_DIR}/baseline_logreg.pkl")
    print(f"  ✅ Saved: {MODEL_DIR}/baseline_logreg.pkl")

    return lr, metrics

# ════════════════════════════════════════════════════════════
# SECTION 06 — MODEL COMPARISON
# ════════════════════════════════════════════════════════════
def compare_models(X_train, X_test, y_train, y_test):
    print("\n" + "="*55)
    print("  SECTION 06 — Model Comparison")
    print("="*55)

    models = {
        "Logistic Regression"  : LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, C=1.0),
        "MultinomialNB"        : MultinomialNB(alpha=0.1),
        "BernoulliNB"          : BernoulliNB(alpha=0.5),
        "LinearSVC"            : CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=RANDOM_STATE, C=1.0)),
    }

    results  = []
    cv_aucs  = []
    roc_data = []

    for name, model in models.items():
        print(f"\n  ── Training: {name} ──")
        t0 = time.time()
        model.fit(X_train, y_train)
        elapsed = time.time() - t0
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]
        m = report_metrics(name, y_test, y_pred, y_proba)
        m['Time'] = elapsed
        results.append(m)

        # 5-fold CV on training set
        print(f"  Running 5-fold CV …")
        cv = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_weighted', n_jobs=-1)
        cv_aucs.append({'Model': name, 'CV_F1': cv.mean(), 'CV_Std': cv.std()})
        print(f"  CV F1: {cv.mean()*100:.2f}% ± {cv.std()*100:.2f}%")

        # ROC curve data
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_data.append((name, fpr, tpr, m['AUC']))

    results_df = pd.DataFrame(results)
    cv_df      = pd.DataFrame(cv_aucs)
    print("\n\n  ── COMPARISON TABLE ──")
    print(results_df[['Model','Accuracy','F1','AUC','Time']].to_string(index=False))

    # ── Plot: Model Comparison Bar ────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = ['#60a5fa','#22d3a5','#fbbf24','#f87171']
    for ax, metric, title in zip(axes, ['Accuracy','F1','AUC'],
                                  ['Accuracy','F1 Score (Weighted)','ROC-AUC']):
        vals = results_df[metric].values * 100 if metric != 'AUC' else results_df[metric].values
        bars = ax.bar(results_df['Model'], vals, color=colors, width=0.5, edgecolor='white')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xticklabels(results_df['Model'], rotation=20, ha='right', fontsize=9)
        ax.set_ylim(min(vals)*0.98, max(vals)*1.015)
        ax.spines[['top','right']].set_visible(False)
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('Model Comparison: Accuracy / F1 / ROC-AUC', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/07_model_comparison_bar.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\n  ✅ Saved: 07_model_comparison_bar.png")

    # ── Plot: ROC Curves ──────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))
    roc_colors = ['#60a5fa','#22d3a5','#fbbf24','#f87171']
    for (name, fpr, tpr, auc_val), c in zip(roc_data, roc_colors):
        ax.plot(fpr, tpr, color=c, lw=2, label=f"{name} (AUC={auc_val:.3f})")
    ax.plot([0,1],[0,1],'--', color='gray', alpha=0.5, label='Random')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/08_roc_curves.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ Saved: 08_roc_curves.png")

    # ── Plot: CV F1 with error bars ───────────────────────────
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(cv_df['Model'], cv_df['CV_F1']*100, yerr=cv_df['CV_Std']*100,
           color=colors, width=0.5, capsize=5, edgecolor='white', error_kw={'ecolor':'white','elinewidth':1.5})
    ax.set_title('5-Fold Cross-Validation F1 Score', fontsize=13, fontweight='bold')
    ax.set_xticklabels(cv_df['Model'], rotation=15, ha='right')
    ax.set_ylabel('CV F1 (%)')
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/09_cv_f1_comparison.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ Saved: 09_cv_f1_comparison.png")

    best_name = results_df.loc[results_df['F1'].idxmax(), 'Model']
    print(f"\n  🏆 Best model: {best_name}")
    return results_df

# ════════════════════════════════════════════════════════════
# SECTION 07 — FINAL MODEL: GridSearchCV + Save
# ════════════════════════════════════════════════════════════
def tune_and_save(X_train, X_test, y_train, y_test):
    print("\n" + "="*55)
    print("  SECTION 07 — GridSearchCV: LinearSVC + TF-IDF Pipeline")
    print("="*55)

    # Load raw text splits for pipeline
    train_df = pd.read_csv("data/train.csv")
    test_df  = pd.read_csv("data/test.csv")
    Xt_raw   = train_df['text'].fillna('')
    Xs_raw   = test_df['text'].fillna('')
    yt       = train_df['label']
    ys       = test_df['label']

    # Build end-to-end pipeline (TF-IDF → LinearSVC)
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=50_000, ngram_range=(1,2),
            min_df=2, max_df=0.95, sublinear_tf=True
        )),
        ('clf', CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=RANDOM_STATE)))
    ])

    param_grid = {
        'tfidf__max_features': [30_000, 50_000],
        'tfidf__ngram_range' : [(1,1), (1,2)],
        'clf__estimator__C': [0.1, 1.0, 5.0],
    }

    print(f"\n  Running GridSearchCV (cv=5) over {2*2*3}={12} combinations …")
    print(f"  This may take 5-10 minutes — go grab a coffee ☕\n")

    gs = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_weighted',
                      n_jobs=-1, verbose=1, refit=True)
    t0 = time.time()
    gs.fit(Xt_raw, yt)
    elapsed = time.time() - t0

    print(f"\n  GridSearch completed in {elapsed/60:.1f} minutes")
    print(f"  Best params : {gs.best_params_}")
    print(f"  Best CV F1  : {gs.best_score_*100:.2f}%")

    # Evaluate best model
    best_model = gs.best_estimator_
    y_pred     = best_model.predict(Xs_raw)
    y_proba    = best_model.predict_proba(Xs_raw)[:, 1]
    metrics    = report_metrics("Final LinearSVC (Tuned)", ys, y_pred, y_proba)

    # ── Final confusion matrix heatmap ────────────────────────
    cm = confusion_matrix(ys, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Negative','Positive'],
                yticklabels=['Negative','Positive'],
                linewidths=0.5, annot_kws={'size':14})
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_title('Final Model — Confusion Matrix Heatmap', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{PLOT_DIR}/10_final_model_confusion_matrix.png", dpi=150, bbox_inches='tight')
    plt.close()
    print(f"\n  ✅ Saved: 10_final_model_confusion_matrix.png")

    # ── Save final model (for Streamlit) ─────────────────────
    model_path = f"{MODEL_DIR}/sentiment_model.pkl"
    joblib.dump(best_model, model_path)
    print(f"  ✅ Saved: {model_path}  (use this in app.py)")

    return best_model, metrics

# ════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════
def main():
    print("=" * 55)
    print("  Phase D — Model Training (05 + 06 + 07)")
    print("=" * 55)

    X_train, X_test, y_train, y_test = load_features()

    # 05 — Baseline
    lr_model, baseline_metrics = train_baseline(X_train, X_test, y_train, y_test)

    # 06 — Comparison
    comparison_df = compare_models(X_train, X_test, y_train, y_test)

    # 07 — Tune & Save
    final_model, final_metrics = tune_and_save(X_train, X_test, y_train, y_test)

    # ── Final Summary ─────────────────────────────────────────
    print("\n\n" + "="*55)
    print("  🏆 PHASE D COMPLETE — RESULTS SUMMARY")
    print("="*55)
    print(f"\n  Baseline  (Logistic Regression) : {baseline_metrics['F1']*100:.2f}% F1")
    print(f"  Final     (LinearSVC Tuned)     : {final_metrics['F1']*100:.2f}% F1")
    print(f"\n  Model saved → {MODEL_DIR}/sentiment_model.pkl")
    f1_val = final_metrics['F1'] * 100    
    print(f"\n  Commit: git add . && git commit -m 'feat: tuned model {f1_val:.1f}% F1 saved'")
    print("\n✅ All model files saved. Ready for Streamlit deployment!")

if __name__ == "__main__":
    main()


  Phase D — Model Training (05 + 06 + 07)
[LOAD] Loading feature matrices from disk …
       X_train: (40000, 50000)  X_test: (10000, 50000)

  SECTION 05 — Baseline: Logistic Regression

  MODEL: Logistic Regression (Baseline)
  Accuracy  : 90.76%
  F1 (wtd)  : 90.76%
  ROC-AUC   : 0.9674
              precision    recall  f1-score   support

    Negative       0.92      0.90      0.91      5000
    Positive       0.90      0.92      0.91      5000

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000


  Training time: 0.4s
  ✅ Saved: 06_baseline_confusion_matrix.png
  ✅ Saved: outputs/models/baseline_logreg.pkl

  SECTION 06 — Model Comparison

  ── Training: Logistic Regression ──

  MODEL: Logistic Regression
  Accuracy  : 90.76%
  F1 (wtd)  : 90.76%
  ROC-AUC   : 0.9674
              precision    recall  f1-score   support

    Negative       0.92      0.90      0.91      